# M2 Task 6 — Colab T4 multilingual embedding cosine

Computes per-pair `emb_name_cos` / `emb_addr_cos` for the blocked candidate pairs using
`intfloat/multilingual-e5-small` (MIT) on a free Tesla T4 (fp16). Inputs are the files exported by
`ber.cli colab-export` (`entities.parquet`, `pairs_{split}.parquet`). Outputs are the small
per-pair parquets imported by `ber.cli colab-import`.

Note: `google.colab.files.upload()` truncates files at ~100 MB; use Google Drive for the 1+ GB
inputs. Model is MIT-licensed, no external data lookup.


In [ ]:
!pip -q install "sentence-transformers>=3.0" 2>&1 | tail -5
import sentence_transformers, torch
print("sentence-transformers", sentence_transformers.__version__, "| torch", torch.__version__, "| cuda", torch.cuda.is_available())


## 1. Inputs

Place `entities.parquet` and `pairs_{train,valfull,test}.parquet` under `/content/colab_in/`
(or `/content/`). This cell validates row counts against the local export.


In [ ]:
import glob, os, pyarrow.parquet as pq

EXPECT = {"entities.parquet": 24229173, "pairs_train.parquet": 30494378,
          "pairs_valfull.parquet": 60912676, "pairs_test.parquet": 250607135}
IN = "/content/colab_in"
os.makedirs(IN, exist_ok=True)
for f in EXPECT:
    hits = glob.glob(f"/content/**/{f}", recursive=True)
    if hits:
        src = hits[0]
        dst = os.path.join(IN, f)
        if os.path.abspath(src) != os.path.abspath(dst):
            os.replace(src, dst)
        meta = pq.ParquetFile(dst)
        print(f, meta.metadata.num_rows, "expected", EXPECT[f],
              "OK" if meta.metadata.num_rows == EXPECT[f] else "MISMATCH")
    else:
        print(f, "MISSING")


## 2. Encode all entities (memmapped, sharded, resumable via marker)


In [ ]:
import json, os, time
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
from sentence_transformers import SentenceTransformer

N_EXPECT = 24229173
DIM = 384
IN = "/content/colab_in"
OUT_NAME = "/content/emb_name.f16.npy"
OUT_ADDR = "/content/emb_addr.f16.npy"
INDEX = "/content/entity_index.parquet"
DONE = "/content/encode_done.json"
BATCH = 131072

if os.path.exists(DONE) and json.load(open(DONE)).get("rows") == N_EXPECT:
    print("already done")
else:
    model = SentenceTransformer("intfloat/multilingual-e5-small", device="cuda")
    model.half()
    model.max_seq_length = 64
    pf = pq.ParquetFile(f"{IN}/entities.parquet")
    n = pf.metadata.num_rows
    name_mm = np.lib.format.open_memmap(OUT_NAME, mode="w+", dtype=np.float16, shape=(n, DIM))
    addr_mm = np.lib.format.open_memmap(OUT_ADDR, mode="w+", dtype=np.float16, shape=(n, DIM))
    writer = pq.ParquetWriter(INDEX, pa.schema([("entity_id", pa.string())]))
    off = 0
    t0 = time.time()
    for bi, batch in enumerate(pf.iter_batches(batch_size=BATCH, columns=["entity_id", "name_norm", "addr_norm"])):
        eids = batch.column("entity_id").to_pylist()
        names = batch.column("name_norm").to_pylist()
        addrs = batch.column("addr_norm").to_pylist()
        en = model.encode(names, batch_size=512, normalize_embeddings=True, convert_to_numpy=True)
        ea = model.encode(addrs, batch_size=512, normalize_embeddings=True, convert_to_numpy=True)
        k = len(eids)
        name_mm[off:off + k] = en.astype(np.float16)
        addr_mm[off:off + k] = ea.astype(np.float16)
        writer.write_table(pa.table({"entity_id": eids}))
        off += k
        if bi % 10 == 0:
            name_mm.flush(); addr_mm.flush()
            rate = off / max(time.time() - t0, 1e-9)
            print(f"{off}/{n} {rate:.0f}/s eta {(n-off)/max(rate,1e-9)/60:.1f}min", flush=True)
    name_mm.flush(); addr_mm.flush(); writer.close()
    json.dump({"rows": off, "dim": DIM}, open(DONE, "w"))
    print("encode done", off)


## 3. Per-pair cosine for each split


In [ ]:
import os, json, time
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import duckdb

DIM = 384
IN = "/content/colab_in"
OUTDIR = "/content/colab_out"
os.makedirs(OUTDIR, exist_ok=True)
os.makedirs("/content/tmp", exist_ok=True)
t = pq.read_table("/content/entity_index.parquet")
t = t.append_column("row", pa.array(np.arange(len(t), dtype=np.int64)))
pq.write_table(t, "/content/entity_index_row.parquet", compression="zstd")
for split in ["train", "valfull", "test"]:
    src = f"{IN}/pairs_{split}.parquet"
    if not os.path.exists(src):
        print("skip missing", split); continue
    out = f"{OUTDIR}/cosine_{split}.parquet"
    if os.path.exists(f"{OUTDIR}/done_{split}.json"):
        print("skip done", split); continue
    En = np.load("/content/emb_name.f16.npy", mmap_mode="r")
    Ea = np.load("/content/emb_addr.f16.npy", mmap_mode="r")
    con = duckdb.connect()
    con.execute("SET memory_limit='6GB'"); con.execute("SET threads=4")
    con.execute("SET temp_directory='/content/tmp'")
    con.execute("CREATE VIEW ei AS SELECT * FROM read_parquet('/content/entity_index_row.parquet')")
    con.execute(f"CREATE VIEW pr AS SELECT * FROM read_parquet('{src}')")
    reader = con.execute("SELECT p.s1_id, p.cand_id, e1.row AS r1, e2.row AS r2 "
                         "FROM pr p JOIN ei e1 ON e1.entity_id=p.s1_id "
                         "JOIN ei e2 ON e2.entity_id=p.cand_id").fetch_record_batch(500000)
    tmp = out + ".tmp"
    writer = None
    total = 0
    t0 = time.time()
    for rb in reader:
        r1 = rb.column("r1").to_numpy(); r2 = rb.column("r2").to_numpy()
        nc = (En[r1].astype(np.float32) * En[r2]).sum(1)
        ac = (Ea[r1].astype(np.float32) * Ea[r2]).sum(1)
        tbl = pa.table({"s1_id": rb.column("s1_id"), "cand_id": rb.column("cand_id"),
                        "emb_name_cos": pa.array(nc.astype(np.float16)),
                        "emb_addr_cos": pa.array(ac.astype(np.float16))})
        if writer is None:
            writer = pq.ParquetWriter(tmp, tbl.schema, compression="zstd")
        writer.write_table(tbl)
        total += len(tbl)
        if total % 5_000_000 < 500000:
            print(split, total, f"{total/max(time.time()-t0,1e-9):.0f}/s", flush=True)
    if writer is not None:
        writer.close()
    os.replace(tmp, out)
    con.close()
    json.dump({"split": split, "rows": total, "bytes": os.path.getsize(out)},
              open(f"{OUTDIR}/done_{split}.json", "w"))
    print("DONE", split, total, os.path.getsize(out))


## 4. Report and package for download


In [ ]:
!ls -lh /content/colab_out
!zip -q -r /content/cosine_out.zip /content/colab_out -x '*.tmp' && ls -lh /content/cosine_out.zip
